# Silver conformance

Validate the bronze → silver pipeline output. Uses `pipelines.conformance` — no duplicated conform logic here.

Expected silver layout:

```
data/silver/locationid=<ID>/year=<YYYY>/<timestamp>_qnxhe_<uuid>
```

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.conformance.conform import SILVER_COLUMNS, build_silver, read_silver

BRONZE_ROOT = ROOT / "data" / "bronze"
SILVER_ROOT = ROOT / "data" / "silver"

## Build silver from bronze

In [2]:
result = build_silver(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT)
print(f"Rows: {result.rows}")
print(f"Files read: {result.files_read}, failed: {result.files_failed}")
print(f"Output: {result.output_path}")

Rows: 432
Files read: 7, failed: 0
Output: D:\CDU\Semester4\Data_Science_Practice\dataguard\data\silver


## Schema and build summary

In [3]:
silver = read_silver(SILVER_ROOT)
build_meta = json.loads((SILVER_ROOT / "_build.json").read_text(encoding="utf-8"))

print("Silver columns:", list(silver.columns))
print("Expected:", SILVER_COLUMNS)
assert list(silver.columns) == SILVER_COLUMNS
print("\n_build.json:")
print(json.dumps(build_meta, indent=2))
print("\nDtypes:")
silver.dtypes

Silver columns: ['sensor_id', 'location', 'datetime', 'latitude', 'longitude', 'parameter', 'unit', 'value']
Expected: ['sensor_id', 'location', 'datetime', 'latitude', 'longitude', 'parameter', 'unit', 'value']

_build.json:
{
  "files_read": 7,
  "files_failed": 0,
  "rows": 432,
  "export_rows": 432,
  "parameters": [
    "pm1",
    "pm25",
    "relativehumidity",
    "temperature",
    "um003"
  ],
  "units": [
    "%",
    "c",
    "particles/cm\u00b3",
    "\u00b5g/m\u00b3"
  ],
  "locations": [
    1544061,
    1601414,
    2455394,
    6430870
  ],
  "years": [
    2026
  ],
  "date_local_min": "2026-01-01",
  "date_local_max": "2026-07-17",
  "failed": []
}

Dtypes:


sensor_id             Int64
location             string
datetime     datetime64[us]
latitude            float64
longitude           float64
parameter            string
unit                 string
value               float64
dtype: object

## Partition layout

In [4]:
exports = [
    p for p in sorted(SILVER_ROOT.rglob("*"))
    if p.is_file() and not p.name.startswith("_")
]
print(f"Export files: {len(exports)}")
for path in exports:
    print(f"  {path.relative_to(SILVER_ROOT)}  ({path.stat().st_size:,} bytes)")

by_location = silver.groupby("location").size().reset_index(name="rows")
by_location

Export files: 4
  locationid=1544061\year=2026\20260902_124141_00088_qnxhe_7f83d75f-7c35-4f80-bd12-c43ff0fdfa35  (6,506 bytes)
  locationid=1601414\year=2026\20260902_124141_00088_qnxhe_9308196d-dc46-41f4-b97e-274e9722f6af  (5,999 bytes)
  locationid=2455394\year=2026\20260902_124141_00088_qnxhe_67f005d4-7ccc-4d99-a02c-990c2e0c35ca  (6,591 bytes)
  locationid=6430870\year=2026\20260902_124141_00088_qnxhe_58761ba8-5f04-41f9-bca2-fdf0c057dd4a  (5,898 bytes)


,location,rows
0,Anzac Memorial-1514036,120
1,Caringbah NSW-1571390,72
2,"Newport, NSW-6400805",120
3,Rozelle-2425184,48
4,Rozelle-4241875,72


## Unit conversion spot-check

In [5]:
pm = silver[silver["parameter"].isin(["pm1", "pm25"])]
print("PM units in silver:", sorted(pm["unit"].unique()))
print("PM parameters:", sorted(pm["parameter"].unique()))
pm.groupby("parameter").agg(
    rows=("value", "count"),
    unit=("unit", "first"),
    min=("value", "min"),
    max=("value", "max"),
)

PM units in silver: ['µg/m³']
PM parameters: ['pm1', 'pm25']


,rows,unit,min,max
parameter,,,,
pm1,72,µg/m³,0.0,9.567857
pm25,96,µg/m³,0.0,16.615476
